<a href="https://colab.research.google.com/github/Hadeel-93/HW-8/blob/main/HW_8_Q1A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import copy
import os

from torch.utils.data import DataLoader

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Dataset settings
data_path = "/content/data-cifar10/"
batch_size = 128
num_epochs = 300

# Transform
transform = transforms.ToTensor()

# Download training dataset
train_dataset = torchvision.datasets.CIFAR10(
    root=data_path,
    train=True,
    download=True,
    transform=transform
)

# Download test dataset
test_dataset = torchvision.datasets.CIFAR10(
    root=data_path,
    train=False,
    download=True,
    transform=transform
)

# Data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

classes = (
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))
print("Number of classes:", len(classes))

Using device: cpu


100%|██████████| 170M/170M [03:43<00:00, 762kB/s]


Training images: 50000
Testing images: 10000
Number of classes: 10


In [2]:
class CNN_Baseline(nn.Module):

    def __init__(self):
        super(CNN_Baseline, self).__init__()

        # First convolution block
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Second convolution block
        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.relu3 = nn.ReLU()

        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):

        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)

        x = self.fc1(x)
        x = self.relu3(x)

        x = self.fc2(x)

        return x


model_a = CNN_Baseline().to(device)

print(model_a)


CNN_Baseline(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=4096, out_features=512, bias=True)
  (relu3): ReLU()
  (fc2): Linear(in_features=512, out_features=10, bias=True)
)


In [3]:
def train_model(model, model_name, epochs=300):

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    eval_losses = []
    eval_accuracies = []

    start_time = time.time()

    for epoch in range(epochs):

        # ---------------- Training ----------------
        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        epoch_train_loss = running_loss / len(train_dataset)
        train_losses.append(epoch_train_loss)

        # ---------------- Evaluation ----------------
        model.eval()

        correct = 0
        total = 0
        eval_running_loss = 0.0

        with torch.no_grad():

            for images, labels in test_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)
                eval_running_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs.data, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        epoch_eval_loss = eval_running_loss / len(test_dataset)
        epoch_accuracy = 100 * correct / total

        eval_losses.append(epoch_eval_loss)
        eval_accuracies.append(epoch_accuracy)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                f"{model_name} | Epoch [{epoch+1}/{epochs}] "
                f"Train Loss: {epoch_train_loss:.4f} | "
                f"Eval Loss: {epoch_eval_loss:.4f} | "
                f"Eval Accuracy: {epoch_accuracy:.2f}%"
            )

    training_time = time.time() - start_time

    # Number of trainable parameters
    total_params = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

    # Save model
    model_path = f"/content/{model_name}.pth"
    torch.save(model.state_dict(), model_path)

    model_size_mb = os.path.getsize(model_path) / (1024 ** 2)

    results = {
        "Model": model_name,
        "Epochs": epochs,
        "Training Time (seconds)": training_time,
        "Final Training Loss": train_losses[-1],
        "Final Evaluation Loss": eval_losses[-1],
        "Final Evaluation Accuracy (%)": eval_accuracies[-1],
        "Parameters": total_params,
        "Model Size (MB)": model_size_mb
    }

    history = {
        "train_losses": train_losses,
        "eval_losses": eval_losses,
        "eval_accuracies": eval_accuracies
    }

    print("\nTraining completed!")
    print("Model:", model_name)
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Final training loss: {train_losses[-1]:.4f}")
    print(f"Final evaluation loss: {eval_losses[-1]:.4f}")
    print(f"Final evaluation accuracy: {eval_accuracies[-1]:.2f}%")
    print(f"Trainable parameters: {total_params:,}")
    print(f"Model size: {model_size_mb:.2f} MB")

    return results, history


In [ ]:
results_a, history_a = train_model(
    model=model_a,
    model_name="CNN_Problem1_A",
    epochs=300
)


CNN_Problem1_A | Epoch [1/300] Train Loss: 1.4751 | Eval Loss: 1.1768 | Eval Accuracy: 58.00%


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history_a["train_losses"], label="Training Loss")
plt.plot(history_a["eval_losses"], label="Evaluation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Problem 1(a): CNN Training and Evaluation Loss")

plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 5))

plt.plot(history_a["eval_accuracies"], label="Evaluation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Problem 1(a): CNN Evaluation Accuracy")

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
class CNN_Additional_Layer(nn.Module):

    def __init__(self):
        super(CNN_Additional_Layer, self).__init__()

        # First convolution block
        self.conv1 = nn.Conv2d(
            3, 32, kernel_size=3, padding=1
        )
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)

        # Second convolution block
        self.conv2 = nn.Conv2d(
            32, 64, kernel_size=3, padding=1
        )
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)

        # Third additional convolution block
        self.conv3 = nn.Conv2d(
            64, 128, kernel_size=3, padding=1
        )
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)

        # Fully connected layers
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.relu4 = nn.ReLU()

        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):

        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.relu3(x)
        x = self.pool3(x)

        x = self.flatten(x)

        x = self.fc1(x)
        x = self.relu4(x)

        x = self.fc2(x)

        return x


model_b = CNN_Additional_Layer().to(device)

print(model_b)


In [ ]:
self.fc1 = nn.Linear(2048, 512)

In [ ]:
results_b, history_b = train_model(
    model=model_b,
    model_name="CNN_Problem1_B",
    epochs=300
)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history_b["train_losses"], label="Training Loss")
plt.plot(history_b["eval_losses"], label="Evaluation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Problem 1(b): CNN Training and Evaluation Loss")

plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 5))

plt.plot(history_b["eval_accuracies"], label="Evaluation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Problem 1(b): CNN Evaluation Accuracy")

plt.legend()
plt.grid(True)
plt.show()


In [ ]:
comparison = pd.DataFrame([results_a, results_b])

print("CNN Model Comparison")
display(comparison)

comparison.to_csv(
    "/content/CNN_Comparison.csv",
    index=False
)


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    history_a["eval_accuracies"],
    label="Problem 1(a): Baseline CNN"
)

plt.plot(
    history_b["eval_accuracies"],
    label="Problem 1(b): Additional Conv Layer"
)

plt.xlabel("Epoch")
plt.ylabel("Evaluation Accuracy (%)")
plt.title("Comparison of CNN Evaluation Accuracy")

plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(12, 5))

plt.plot(
    history_a["train_losses"],
    label="Problem 1(a): Baseline CNN Training Loss"
)

plt.plot(
    history_b["train_losses"],
    label="Problem 1(b): Extended CNN Training Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Comparison of CNN Training Loss")

plt.legend()
plt.grid(True)
plt.show()


In [ ]:
print("Problem 1(a)")
print("Final Training Loss:", history_a["train_losses"][-1])
print("Final Evaluation Loss:", history_a["eval_losses"][-1])

print("\nProblem 1(b)")
print("Final Training Loss:", history_b["train_losses"][-1])
print("Final Evaluation Loss:", history_b["eval_losses"][-1])

gap_a = history_a["eval_losses"][-1] - history_a["train_losses"][-1]
gap_b = history_b["eval_losses"][-1] - history_b["train_losses"][-1]

print("\nLoss Gap (Evaluation - Training)")
print("Problem 1(a):", round(gap_a, 4))
print("Problem 1(b):", round(gap_b, 4))